In [11]:
!pip install flask flask-sqlalchemy flask-login pyngrok flask-ngrok

In [2]:
%%writefile models.py
from app import db, login
from flask_login import UserMixin
from sqlalchemy.orm import Mapped, mapped_column, relationship
from datetime import datetime
from typing import List

class User(UserMixin, db.Model):
    __tablename__ = 'users'
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(unique=True)
    password: Mapped[str] = mapped_column()
    foto_perfil: Mapped[str] = mapped_column(nullable=True) # Requisito Item 3
    bio: Mapped[str] = mapped_column(nullable=True)         # Requisito Item 3

    # Relacionamento: um usuário tem muitos posts
    posts: Mapped[List["Post"]] = relationship(back_populates="author")

class Post(db.Model):
    id: Mapped[int] = mapped_column(primary_key=True)
    body: Mapped[str] = mapped_column()
    timestamp: Mapped[datetime] = mapped_column(default=datetime.utcnow)
    user_id: Mapped[int] = mapped_column(db.ForeignKey('users.id')) # Chave estrangeira

    # Relacionamento: cada post tem um autor
    author: Mapped["User"] = relationship(back_populates="posts")

@login.user_loader
def load_user(id):
    return db.session.get(User, int(id)) #

Writing models.py


In [3]:
%%writefile alquimias.py
from app import db
from models import User, Post

def validate_user_password(username, password):
    user = db.session.scalar(db.select(User).filter_by(username=username))
    if user and user.password == password:
        return user
    return None

def user_exists(username):
    return db.session.scalar(db.select(User).filter_by(username=username)) is not None

def create_user(username, password, foto, bio):
    new_user = User(username=username, password=password, foto_perfil=foto, bio=bio)
    db.session.add(new_user)
    db.session.commit()
    return new_user

def create_post(body, user): # Requisito Item 4.3
    new_post = Post(body=body, author=user)
    db.session.add(new_post)
    db.session.commit()

def get_timeline(): # Requisito Item 4.3
    return db.session.scalars(db.select(Post).order_by(Post.timestamp.desc()).limit(5)).all()

Writing alquimias.py


In [7]:
# Célula de Teste Manual
from app import app, db
from models import User, Post
from datetime import datetime

with app.app_context():
    db.create_all() # Garante que as tabelas existam
    # Teste de listagem
    user = db.session.get(User, 1)
    if user:
        for post in user.posts:
            print(f"Post ID: {post.id} | Conteúdo: {post.body} | Autor: {post.author.username}")

In [6]:
%%writefile app.py
from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager

app = Flask(__name__)
app.config['SECRET_KEY'] = 'sua_chave_secreta_aqui'  # ATENÇÃO: Substitua por uma chave secreta forte e única!
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///site.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
db = SQLAlchemy(app)
login = LoginManager(app)
login.login_view = 'login' # Define a view para onde o usuário será redirecionado se não estiver logado

Writing app.py


In [8]:
%%writefile routes.py
from flask import render_template, redirect, url_for, request
from flask_login import current_user, login_user, logout_user, login_required
from app import app
import alquimias

@app.route('/')
@login_required
def index():
    posts = alquimias.get_timeline() # Busca os 5 posts recentes
    return render_template('index.html', user=current_user, posts=posts)

@app.route('/login', methods=['GET', 'POST'])
def login():
    if current_user.is_authenticated:
        return redirect(url_for('index'))
    if request.method == 'POST':
        username = request.form['username'].lower()
        password = request.form['password'].lower()
        user = alquimias.validate_user_password(username, password)
        if user:
            login_user(user, remember=True)
            return redirect(url_for('index'))
    return render_template('login.html')

@app.route('/cadastro', methods=['GET', 'POST'])
def cadastro():
    if request.method == 'POST':
        username = request.form['username'].lower()
        if alquimias.user_exists(username):
            return redirect(url_for('login'))

        password = request.form['password'].lower()
        # Capturando novos campos (Requisito Item 3)
        foto = request.form.get('foto')
        bio = request.form.get('bio')

        user = alquimias.create_user(username, password, foto, bio)
        login_user(user)
        return redirect(url_for('index'))
    return render_template('cadastro.html')

@app.route('/post', methods=['GET', 'POST'])
@login_required
def post():
    if request.method == 'POST':
        body = request.form.get('body')
        alquimias.create_post(body, current_user)
        return redirect(url_for('index'))
    return render_template('post.html')

@app.route('/logout')
def logout():
    logout_user()
    return redirect(url_for('index'))

Writing routes.py


In [9]:
import os
if not os.path.exists('templates'):
    os.makedirs('templates')

# Exemplo de cadastro.html (Requisito Item 3)
with open('templates/cadastro.html', 'w') as f:
    f.write('''
    <form method="post">
        <input name="username" placeholder="Usuário" required>
        <input name="password" type="password" placeholder="Senha" required>
        <input name="foto" placeholder="URL da Foto de Perfil">
        <textarea name="bio" placeholder="Sua Bio"></textarea>
        <button type="submit">Cadastrar</button>
    </form>
    ''')

# Exemplo de post.html (Requisito Item 4.3)
with open('templates/post.html', 'w') as f:
    f.write('''
    <form method="post">
        <textarea name="body" placeholder="O que está pensando?" required></textarea>
        <button type="submit">Publicar</button>
    </form>
    ''')

In [ ]:
from flask_ngrok import run_with_ngrok
from pyngrok import ngrok
from app import app, db

# Substitua pelo seu token do site ngrok.com
ngrok.set_auth_token("SEU_TOKEN_AQUI")

# Cria o banco de dados antes de rodar
with app.app_context():
    db.create_all()

run_with_ngrok(app)

if __name__ == '__main__':
    app.run()

 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Exception in thread Thread-8:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
Exception in thread Thread-6:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
Thread-7:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12